# Модуль-ноутбук: `features`

Стейтлес-інженерія pre-post ознак + **стіна проти витоку** (`assert_no_leakage`). Той самий код на train та inference.

**Залежності:** `%run` 00_config.ipynb

In [ ]:
%run 00_config.ipynb

In [ ]:
"""Stateless, pre-post-only feature engineering.

THE leakage firewall lives here. Two guarantees, both tested:
  1. `engineer_features` only reads columns in `config.PREPOST_COLS`; it never looks at
     a post-hoc metric. `assert_no_leakage` enforces that the OUTPUT contains no post-hoc
     column name either.
  2. The transform is *stateless* — the identical function is used at training time and at
     inference time, so a candidate is featurized exactly like a training row.

All text work is done with plain-Python string ops (not the pandas `.str` accessor) so the
behaviour is identical on pandas 2.x (numpy strings) and pandas 3.x (pyarrow strings).
"""

import math
import re

import numpy as np
import pandas as pd


# ----------------------------------------------------------------------------- text lexicons
# Tiny, transparent sentiment lexicon (heuristic — documented in PLAN/README). Not a
# replacement for a real model; just a cheap signal for "tone" of the caption.
_POS_WORDS = {
    "love", "best", "amazing", "awesome", "great", "happy", "win", "winner", "free",
    "easy", "perfect", "beautiful", "wow", "incredible", "fun", "good", "new", "cool",
    "favorite", "favourite", "epic", "insane", "satisfying", "magic", "viral", "lol",
    "haha", "yes", "tasty", "delicious", "cute", "genius", "hack", "secret",
}
_NEG_WORDS = {
    "hate", "worst", "bad", "sad", "angry", "fail", "broke", "broken", "ugly", "boring",
    "terrible", "awful", "cringe", "gross", "scary", "fake", "wrong", "lost", "problem",
    "annoying", "disgusting", "trash", "stupid", "hard", "difficult", "pain",
}
_POS_EMOJI = set("😀😁😂🤣😊😍🥰😎🔥💯👍❤️✨🎉🥳😋🤩💖👏🙌😆")
_NEG_EMOJI = set("😡😠😢😭😞👎💔😒🤬🙄😤😩😱")

_CTA_PATTERNS = [
    "link in bio", "follow", "subscribe", "comment below", "comment", "tag a", "tag your",
    "duet", "stitch", "check out", "swipe", "click", "join", "giveaway", "sign up",
    "like and", "share this", "save this", "drop a", "let me know", "dm me",
]
_URL_RE = re.compile(r"(https?://|www\.|\b\w+\.(com|net|org|io|me)\b)", re.IGNORECASE)
_HASHTAG_RE = re.compile(r"#\w+", re.UNICODE)
_MENTION_RE = re.compile(r"@[\w.]+", re.UNICODE)
_EMOJI_RE = re.compile(
    "["
    "\U0001F300-\U0001FAFF"   # symbols, pictographs, emoticons, supplemental
    "\U00002600-\U000027BF"   # misc symbols + dingbats
    "\U0001F1E6-\U0001F1FF"   # regional indicators (flags)
    "\U00002B00-\U00002BFF"   # misc symbols & arrows
    "\U00002190-\U000021FF"   # arrows
    "]",
    flags=re.UNICODE,
)
_WORD_RE = re.compile(r"[^\W\d_]{2,}", re.UNICODE)  # alphabetic words, len>=2

# Frozen, ordered feature contract. data_prep / train / inference all rely on this order.
FEATURE_COLUMNS = [
    # --- text ---
    "char_len", "word_len", "n_hashtags", "n_mentions", "has_question", "has_exclam",
    "n_emoji", "has_emoji", "has_url", "has_cta", "allcaps_word_ratio", "digit_ratio",
    "mean_word_len", "caption_is_empty", "sentiment",
    # --- duration --- (no low-support missing flag: it destabilised the linear model;
    # missing duration is median-imputed and the inference layer shrinks confidence instead)
    "duration_s",
    "dur_very_short", "dur_short", "dur_mid", "dur_long", "dur_very_long",
    # --- time (UTC assumption); cyclical sin/cos so the linear model treats hour/day/
    #     month as periodic (not as a spurious linear ramp / drift proxy) ---
    "hour_sin", "hour_cos", "dow_sin", "dow_cos", "is_weekend", "month_sin", "month_cos",
    "pod_night", "pod_morning", "pod_afternoon", "pod_evening",
]

# A compact, human-friendly label per feature for the UI "factors" panel.
FEATURE_LABELS = {
    "char_len": "caption length (chars)", "word_len": "caption length (words)",
    "n_hashtags": "# hashtags", "n_mentions": "# mentions", "has_question": "asks a question",
    "has_exclam": "has exclamation", "n_emoji": "# emoji", "has_emoji": "uses emoji",
    "has_url": "contains a link", "has_cta": "has call-to-action",
    "allcaps_word_ratio": "ALL-CAPS words ratio", "digit_ratio": "digit ratio",
    "mean_word_len": "avg word length", "caption_is_empty": "empty caption",
    "sentiment": "caption sentiment", "duration_s": "duration (s)",
    "duration_missing": "duration missing", "dur_very_short": "very short (<7s)",
    "dur_short": "short (7–15s)", "dur_mid": "medium (15–60s)", "dur_long": "long (60–180s)",
    "dur_very_long": "very long (>180s)",
    "hour_sin": "post hour (cyclical)", "hour_cos": "post hour (cyclical)",
    "dow_sin": "day of week (cyclical)", "dow_cos": "day of week (cyclical)",
    "is_weekend": "weekend", "month_sin": "month (seasonal)", "month_cos": "month (seasonal)",
    "pod_night": "posted at night", "pod_morning": "posted in morning",
    "pod_afternoon": "posted in afternoon", "pod_evening": "posted in evening",
}

In [ ]:
def _safe_str(x) -> str:
    if x is None:
        return ""
    try:
        if pd.isna(x):
            return ""
    except (TypeError, ValueError):
        pass
    return str(x)

In [ ]:
def _text_features(caption: str) -> dict:
    """All caption-derived features for one string. Degrades gracefully on ''."""
    c = _safe_str(caption)
    char_len = len(c)          # raw, used for ratios below
    words = c.split()
    word_len = len(words)      # raw, used for ratios below
    alpha_words = _WORD_RE.findall(c)
    n_caps = sum(1 for w in alpha_words if w.isupper())
    pos = sum(1 for w in words if w.strip(".,!?#@").lower() in _POS_WORDS)
    neg = sum(1 for w in words if w.strip(".,!?#@").lower() in _NEG_WORDS)
    pos += sum(1 for ch in c if ch in _POS_EMOJI)
    neg += sum(1 for ch in c if ch in _NEG_EMOJI)
    sentiment = (pos - neg) / (pos + neg + 1.0)
    low = c.lower()
    mean_word_len = (sum(len(w) for w in words) / word_len) if word_len else 0.0
    # All unbounded features are winsorized to realistic ranges so an out-of-distribution
    # caption (e.g. one 2000-char "word", or 500 emoji) cannot saturate the linear model.
    return {
        "char_len": float(min(char_len, config.CHAR_LEN_CLIP)),
        "word_len": float(min(word_len, config.WORD_LEN_CLIP)),
        "n_hashtags": float(min(len(_HASHTAG_RE.findall(c)), 15)),
        "n_mentions": float(min(len(_MENTION_RE.findall(c)), 15)),
        "has_question": float("?" in c),
        "has_exclam": float("!" in c),
        "n_emoji": float(min(len(_EMOJI_RE.findall(c)), 20)),
        "has_emoji": float(bool(_EMOJI_RE.search(c))),
        "has_url": float(bool(_URL_RE.search(c))),
        "has_cta": float(any(p in low for p in _CTA_PATTERNS)),
        "allcaps_word_ratio": (n_caps / len(alpha_words)) if alpha_words else 0.0,
        "digit_ratio": (sum(ch.isdigit() for ch in c) / char_len) if char_len else 0.0,
        "mean_word_len": float(min(mean_word_len, 30.0)),
        "caption_is_empty": float(char_len == 0),
        "sentiment": float(sentiment),
    }

In [ ]:
def _duration_features(seconds: float) -> dict:
    missing = seconds is None or (isinstance(seconds, float) and np.isnan(seconds)) or seconds <= 0
    val = np.nan if missing else float(min(max(seconds, config.DURATION_CLIP[0]), config.DURATION_CLIP[1]))
    s = np.nan if missing else float(seconds)
    return {
        "duration_s": val,
        "duration_missing": float(missing),
        "dur_very_short": float((not missing) and s < 7),
        "dur_short": float((not missing) and 7 <= s < 15),
        "dur_mid": float((not missing) and 15 <= s < 60),
        "dur_long": float((not missing) and 60 <= s < 180),
        "dur_very_long": float((not missing) and s >= 180),
    }

In [ ]:
def _time_features(epoch: float) -> dict:
    nan_block = {
        "hour_sin": np.nan, "hour_cos": np.nan, "dow_sin": np.nan, "dow_cos": np.nan,
        "is_weekend": np.nan, "month_sin": np.nan, "month_cos": np.nan,
        "time_valid": 0.0, "pod_night": 0.0, "pod_morning": 0.0,
        "pod_afternoon": 0.0, "pod_evening": 0.0,
    }
    valid = (
        epoch is not None and not (isinstance(epoch, float) and np.isnan(epoch))
        and np.isfinite(epoch)
        and config.MIN_PLAUSIBLE_EPOCH <= epoch <= config.MAX_PLAUSIBLE_EPOCH
    )
    if not valid:
        return nan_block
    try:  # last-resort guard: never let timestamp math escape the shared feature path
        ts = pd.Timestamp(float(epoch), unit="s", tz=config.ASSUME_TIMEZONE)
        hour, dow, month = ts.hour, ts.dayofweek, ts.month
    except Exception:
        return nan_block
    pod = "night" if hour < 6 else "morning" if hour < 12 else "afternoon" if hour < 18 else "evening"
    return {
        "hour_sin": math.sin(2 * math.pi * hour / 24), "hour_cos": math.cos(2 * math.pi * hour / 24),
        "dow_sin": math.sin(2 * math.pi * dow / 7), "dow_cos": math.cos(2 * math.pi * dow / 7),
        "is_weekend": float(dow >= 5),
        "month_sin": math.sin(2 * math.pi * (month - 1) / 12),
        "month_cos": math.cos(2 * math.pi * (month - 1) / 12),
        "time_valid": 1.0,
        "pod_night": float(pod == "night"), "pod_morning": float(pod == "morning"),
        "pod_afternoon": float(pod == "afternoon"), "pod_evening": float(pod == "evening"),
    }

In [ ]:
def engineer_features(df: pd.DataFrame) -> pd.DataFrame:
    """Map raw rows -> the frozen pre-post feature matrix. Stateless & leakage-safe."""
    captions = df["description"].tolist() if "description" in df else [""] * len(df)
    durations = (
        pd.to_numeric(df["duration"], errors="coerce").tolist()
        if "duration" in df else [np.nan] * len(df)
    )
    times = (
        pd.to_numeric(df[config.TIME_COL], errors="coerce").tolist()
        if config.TIME_COL in df else [np.nan] * len(df)
    )

    rows = []
    for cap, dur, t in zip(captions, durations, times):
        row = {}
        row.update(_text_features(cap))
        row.update(_duration_features(dur))
        row.update(_time_features(t))
        rows.append(row)

    out = pd.DataFrame(rows, columns=FEATURE_COLUMNS, index=df.index)
    assert_no_leakage(out)
    return out

In [ ]:
def assert_no_leakage(feature_df: pd.DataFrame) -> None:
    """Hard guard: the feature matrix must contain no post-hoc / identifier columns."""
    forbidden = set(config.POSTHOC_COLS) | set(config.ID_COLS)
    leaked = forbidden.intersection(feature_df.columns)
    if leaked:
        raise AssertionError(f"LEAKAGE: post-hoc columns found in features: {sorted(leaked)}")
    extra = set(feature_df.columns) - set(FEATURE_COLUMNS)
    missing = set(FEATURE_COLUMNS) - set(feature_df.columns)
    if extra or missing:
        raise AssertionError(f"feature contract mismatch (extra={extra}, missing={missing})")

In [ ]:
from types import SimpleNamespace
features = SimpleNamespace(
    engineer_features=engineer_features,
    assert_no_leakage=assert_no_leakage,
    FEATURE_COLUMNS=FEATURE_COLUMNS,
    FEATURE_LABELS=FEATURE_LABELS,
)

### Перевірка / демо

In [ ]:
import pandas as pd
_demo = pd.DataFrame([{'description':'POV 🔥 #fyp','duration':18,'create_time':1_700_000_000}])
print('фічей:', len(features.FEATURE_COLUMNS))
features.engineer_features(_demo).T.head(8)